In [1]:
import numpy as np
import pandas as pd
from datasets import ClassLabel, load_dataset

/Users/arslanniazgulov/Documents/nlp_projects/ticket-routing-service/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import os
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
RANDOM_SEED = 42

os.makedirs(PROJECT_ROOT / "data", exist_ok=True)

In [3]:
dataset = load_dataset("AutoIntent/banking77", cache_dir=PROJECT_ROOT / "data")

In [4]:
import json

with open("id2label.json", "r") as f:
    id2label = json.load(f)

id2label = {int(i): label for i, label in id2label.items()}

In [5]:
train_dataset = dataset["train"]
test_dataset = dataset["test"]

In [6]:
train_dataset_casted = train_dataset.cast_column(
    "label", ClassLabel(num_classes=len(id2label.keys()), names=list(id2label.values()))
)
train_val_dataset = train_dataset_casted.train_test_split(
    test_size=0.2, shuffle=True, stratify_by_column="label", seed=RANDOM_SEED
)

train_dataset_splited = train_val_dataset["train"]
val_dataset_splited = train_val_dataset["test"]

In [7]:
UTTERANCE = "utterance"
LABEL = "label"

### Анализ Данных

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

In [ ]:
counter = Counter(train_dataset_splited["label"])

print("Самый популярный класс: ", counter.most_common(5))
print("Самый редкий класс: ", counter.most_common()[:-5:-1])

In [ ]:
def add_len(ex):
    ex["length"] = len(ex[UTTERANCE])
    return ex


train_dataset_splited = train_dataset_splited.map(add_len)

In [ ]:
length_counter = Counter(train_dataset_splited["length"])

In [ ]:
print("Самые частые length (длина, кол-во):", *length_counter.most_common(5))
print("Самые редкие length (длина, кол-во):", *length_counter.most_common()[:-10:-1])

In [ ]:
label_length_df = pd.DataFrame(
    {"label": train_dataset_splited["label"], "length": train_dataset_splited["length"]}
)

In [ ]:
label_length_df.groupby("label").aggregate(["min", "max", "mean", "count"])

# BaseLine

In [ ]:
X_train = train_dataset_splited[UTTERANCE]
y_train = train_dataset_splited[LABEL]

X_val = val_dataset_splited[UTTERANCE]
y_val = val_dataset_splited[LABEL]

X_test = test_dataset[UTTERANCE]
y_test = test_dataset[LABEL]

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.pipeline import Pipeline

In [ ]:
baseline = Pipeline(
    [
        ("tfidf", TfidfVectorizer(min_df=1, max_df=0.9)),
        ("log_reg", LogisticRegression()),
    ]
)

In [ ]:
baseline.fit(X_train, y_train)
y_pred = baseline.predict(X_test)
print("macro F1:   ", f1_score(y_test, y_pred, average="macro"))
print("micro F1:   ", f1_score(y_test, y_pred, average="micro"))
print("weighted F1:", f1_score(y_test, y_pred, average="weighted"))
print("accuracy:   ", accuracy_score(y_test, y_pred))
# print(classification_report(y_test, y_pred, zero_division=0))

## Байесовский перебор параметров

In [ ]:
import optuna
from sklearn.model_selection import cross_val_score

X = train_dataset_splited[UTTERANCE]
y = train_dataset_splited[LABEL]

def objective(trial):

    max_n = trial.suggest_int("max_ngram", 1, 2)
    ngram_range = (1, max_n)
    sublinear_tf = trial.suggest_categorical("sublinear_tf", [True, False])
    max_features = trial.suggest_categorical("max_features", [5000, 10000, 20000, None])
    min_df = trial.suggest_int("min_df", 1, 5)
    C = trial.suggest_float("C", 1e-2, 1e2, log=True)
    class_weight = trial.suggest_categorical("class_weight", [None, "balanced"])
    
    
    pipeline = Pipeline(
        [
            (
                "tfidf",
                TfidfVectorizer(
                    ngram_range=ngram_range,
                    sublinear_tf=sublinear_tf,
                    max_features=max_features,
                    min_df=min_df,
                    max_df=0.9
                ),
            ),
            (
                "log_reg",
                LogisticRegression(
                    C=C,
                    class_weight=class_weight,
                    solver="saga",       # saga быстрый и стабильный для текстов
                    max_iter=1000,       # чтобы модель точно сошлась
                    random_state=42
                ),
            ),
        ]
    )
    
    score = cross_val_score(pipeline, X, y, cv=5, scoring="f1_macro", n_jobs=-1).mean()
    return score


In [ ]:
student = optuna.create_study(direction="maximize", study_name="tfidf+logreg")
student.optimize(objective, n_trials=30,)

In [ ]:
best_params = student.best_params
print(f"Лучший F1 на валидации: {student.best_value:.4f}")


final_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        ngram_range=(1, best_params.get("max_ngram", 1)),
        sublinear_tf=best_params["sublinear_tf"],
        max_features=best_params["max_features"],
        min_df=best_params["min_df"],
        max_df=0.9
    )),
    ("log_reg", LogisticRegression(
        C=best_params["C"],
        class_weight=best_params["class_weight"],
        solver="saga",
        max_iter=1000,
        random_state=42
    ))
])

final_pipeline.fit(X, y)

Лучший F1 на валидации: 0.8772


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('tfidf', ...), ('log_reg', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](77,)","[ 0, 1, 2,...,74,75,76]"
,"max_df max_df: float or int, default=1.0When building the vocabulary ignore terms that have a documentfrequency strictly higher than the given threshold (corpus-specificstop words).If float in range [0.0, 1.0], the parameter represents a proportion ofdocuments, integer absolute counts.This parameter is ignored if vocabulary is not None.",0.9
,"max_features max_features: int, default=NoneIf not None, build a vocabulary that only consider the top`max_features` ordered by term frequency across the corpus.Otherwise, all features are used.This parameter is ignored if vocabulary is not None.",20000
,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict', meaning that a UnicodeDecodeError will be raised. Othervalues are 'ignore' and 'replace'.",'strict'


## MlFlow Logging

In [ ]:
from sklearn.metrics import classification_report, f1_score

# Предсказание на отложенном тесте
y_pred = final_pipeline.predict(X_test)

test_f1 = f1_score(y_test, y_pred, average="macro")
print(f"Финальный F1-macro на Test: {test_f1:.4f}")

Финальный F1-macro на Test: 0.8901


In [29]:
import mlflow
import mlflow.sklearn

# 1. Подключаемся к нашей базе SQLite и эксперименту
mlflow.set_tracking_uri(f"sqlite:///{PROJECT_ROOT / 'mlflow.db'}")
mlflow.set_experiment("banking77-intent")

# 2. Финальный запуск
with mlflow.start_run(run_name="final_best_model_baseline"):
    # Логируем лучшие параметры и метрику
    mlflow.log_params(best_params)
    mlflow.log_metric("test_f1_macro", test_f1)
    
    # Сохраняем готовую модель Pipeline
    mlflow.sklearn.log_model(final_pipeline, name="model")
    
    # Прикрепляем файл с названиями категорий
    mlflow.log_artifact("id2label.json")

print(" Модель и артефакты успешно сохранены в MLflow!")

2026/11/14 10:54:38 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


 Модель и артефакты успешно сохранены в MLflow!


# Трансформер based

In [31]:
import os

import mlflow
from transformers import (
    DataCollatorWithPadding,
    DistilBertForSequenceClassification,
    DistilBertTokenizer,
    Trainer,
    TrainingArguments,
)

os.environ["MLFLOW_ENABLE_ASYNC_LOGGING"] = "false"
mlflow.end_run()
os.environ["MLFLOW_TRACKING_URI"] = f"sqlite:///{PROJECT_ROOT / 'mlflow.db'}"
os.environ["MLFLOW_EXPERIMENT_NAME"] = "banking77-intent"

In [9]:
MODEL_NAME = "distilbert-base-uncased"
NUM_CLASSES = 77
MAX_LEN = 100

In [10]:
label2id = {label: i for i, label in id2label.items()}

In [11]:
tokenizer = DistilBertTokenizer.from_pretrained(
    MODEL_NAME, cache_dir=PROJECT_ROOT / "transformers" / MODEL_NAME
)

model = DistilBertForSequenceClassification.from_pretrained(
    MODEL_NAME,
    cache_dir=PROJECT_ROOT / "transformers" / MODEL_NAME / "model",
    num_labels=NUM_CLASSES,
    id2label=id2label,
    label2id=label2id,
    attn_implementation="eager"
)

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 7426.57it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [12]:
print(model.config.num_labels)
print(model.config.id2label[0])
print(model.classifier.out_features)

77
activate_my_card
77


In [13]:
def tokenize(batch):
    return tokenizer(batch[UTTERANCE], truncation=True, max_length=MAX_LEN)

In [14]:
val_dataset_splited

Dataset({
    features: ['utterance', 'label'],
    num_rows: 2001
})

In [ ]:
tok_train = train_dataset_splited.map(tokenize, batched=True)
tok_val = val_dataset_splited.map(tokenize, batched=True)

In [43]:
tok_test = test_dataset.map(tokenize, batched=True)

Map: 100%|██████████| 3080/3080 [00:00<00:00, 4047.84 examples/s]


In [16]:
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score


def compute_metrics(eval_preds):
    logits, labels = eval_preds
    preds = np.argmax(logits, axis=-1)
    return {
        "macro_f1": f1_score(labels, preds, average="macro", zero_division=0),
        "weighted_f1": f1_score(labels, preds, average="weighted", zero_division=0),
        "recall_macro": recall_score(labels, preds, average="macro", zero_division=0),
        "precision_macro": precision_score(
            labels, preds, average="macro", zero_division=0
        ),
        "accuracy": accuracy_score(labels, preds),
    }

In [17]:
import torch
from sklearn.utils.class_weight import compute_class_weight
from transformers import Trainer

cw = compute_class_weight(
    "balanced", classes=np.unique(tok_train["label"]), y=tok_train["label"]
)
class_weights = torch.tensor(cw, dtype=torch.float)
class_weights

tensor([0.8183, 1.1809, 1.0289, 1.4846, 1.0188, 0.7586, 0.7167, 0.8314, 0.8248,
        1.0090, 2.2111, 0.8518, 1.1547, 0.9362, 1.1547, 0.6975, 0.7755, 0.7755,
        2.1209, 0.7370, 0.8119, 1.0604, 1.5061, 3.7115, 1.0090, 0.8518, 0.7531,
        0.9804, 0.7167, 1.0714, 1.0714, 1.0714, 1.1547, 1.1056, 0.7814, 0.9447,
        1.0289, 1.3323, 1.2226, 1.0090, 1.3323, 1.5746, 1.0714, 1.0825, 1.2372,
        0.8183, 0.9116, 0.8733, 0.8807, 1.1296, 1.3674, 0.7994, 0.7698, 0.8056,
        1.0090, 1.2084, 1.1677, 1.1420, 1.1420, 0.8959, 1.3323, 0.8882, 1.2673,
        0.7423, 0.7531, 1.1547, 0.7586, 1.0188, 1.2673, 1.2521, 1.1547, 1.0289,
        3.1492, 0.9622, 1.0714, 0.7217, 0.7994])

In [18]:
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        loss = torch.nn.CrossEntropyLoss(
            weight=class_weights.to(outputs.logits.device)
        )(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss

In [ ]:
training_args = TrainingArguments(
    output_dir=f"runs/hf/{MODEL_NAME}",
    learning_rate=3e-5,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    per_device_eval_batch_size=64,
    num_train_epochs=5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    logging_steps=50,
    report_to="mlflow",
    seed=RANDOM_SEED,
)

In [33]:
trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=tok_train,
    eval_dataset=tok_val,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
)

In [34]:
trainer.train()

/Users/arslanniazgulov/Documents/nlp_projects/ticket-routing-service/.venv/lib/python3.14/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Macro F1,Weighted F1,Recall Macro,Precision Macro,Accuracy
1,1.369154,0.423616,0.886060,0.885626,0.884765,0.896876,0.885557
2,0.855978,0.367964,0.903631,0.903825,0.902755,0.912065,0.904048
3,0.359588,0.380680,0.908375,0.909375,0.907137,0.917700,0.909045
4,0.238971,0.364190,0.911862,0.911889,0.910323,0.919044,0.912044
5,0.140135,0.369762,0.911338,0.911804,0.909762,0.918963,0.912044


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.23it/s]
/Users/arslanniazgulov/Documents/nlp_projects/ticket-routing-service/.venv/lib/python3.14/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.25it/s]
/Users/arslanniazgulov/Documents/nlp_projects/ticket-routing-service/.venv/lib/python3.14/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.09it/s]
/Users/arslanniazgulov/Documents/nlp_projects/ticket-routing-service/.venv/lib/python3.14/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pin

TrainOutput(global_step=2505, training_loss=0.6501374520227581, metrics={'train_runtime': 1293.6515, 'train_samples_per_second': 30.928, 'train_steps_per_second': 1.936, 'total_flos': 267258095440800.0, 'train_loss': 0.6501374520227581, 'epoch': 5.0})

In [44]:
preds = trainer.predict(tok_test)

/Users/arslanniazgulov/Documents/nlp_projects/ticket-routing-service/.venv/lib/python3.14/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


In [47]:
preds.metrics

{'test_loss': 0.3477918803691864,
 'test_macro_f1': 0.9175252240656748,
 'test_weighted_f1': 0.9175252240656746,
 'test_recall_macro': 0.9175324675324674,
 'test_precision_macro': 0.9208996170858267,
 'test_accuracy': 0.9175324675324675,
 'test_runtime': 32.9055,
 'test_samples_per_second': 93.601,
 'test_steps_per_second': 1.489}

### LoRa

In [52]:
from peft import LoraConfig, get_peft_model, TaskType

In [67]:
model = DistilBertForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_CLASSES,
    cache_dir=PROJECT_ROOT / "transformers" / MODEL_NAME / "model",
)

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 4970.44it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [68]:
lora_config = LoraConfig(
    r=32,
    lora_alpha=32,
    bias="none",
    task_type=TaskType.SEQ_CLS,
    target_modules=[
        "q_lin",  # Query в attention
        "k_lin",  # Key в attention
        "v_lin",  # Value в attention
        "out_lin",  # Output projection
        "lin1",  # FFN intermediate (вместо w1)
        "lin2",  # FFN output (вместо w2)
    ],
    modules_to_save=[
        'pre_classifier',
        'classifier'
    ]
)

model_lora = get_peft_model(model, lora_config)
model_lora.print_trainable_parameters()

trainable params: 3,304,013 || all params: 70,316,698 || trainable%: 4.6988


/Users/arslanniazgulov/Documents/nlp_projects/ticket-routing-service/.venv/lib/python3.14/site-packages/transformers/modeling_utils.py:3303: FutureWarning: `is_gradient_checkpointing` is deprecated and will be removed in v6. Use `is_activation_checkpointing` instead.
  warnings.warn(


In [72]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir=f"runs/hf/{MODEL_NAME}",
    learning_rate=2e-4,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=64,
    gradient_accumulation_steps=8,
    num_train_epochs=5,
    weight_decay=0.01,
    warmup_steps=0.25,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    logging_steps=50,
    report_to="mlflow",
    run_name="distil-bert-lora-lr-fixed",
    seed=RANDOM_SEED,
    lr_scheduler_type="cosine"
)

trainer = WeightedTrainer(
    model=model_lora,
    args=training_args,
    train_dataset=tok_train,
    eval_dataset=tok_val,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
)

In [73]:
trainer.train()

/Users/arslanniazgulov/Documents/nlp_projects/ticket-routing-service/.venv/lib/python3.14/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Macro F1,Weighted F1,Recall Macro,Precision Macro,Accuracy
1,2.917767,0.458282,0.873129,0.870897,0.872251,0.885942,0.872064
2,2.287352,0.418662,0.875214,0.876295,0.876538,0.888000,0.878061
3,1.881509,0.367645,0.897301,0.896459,0.898073,0.904809,0.897051
4,1.321450,0.344568,0.906523,0.905205,0.906253,0.911586,0.905547
5,1.196441,0.341450,0.906714,0.906363,0.906905,0.911952,0.906547


/Users/arslanniazgulov/Documents/nlp_projects/ticket-routing-service/.venv/lib/python3.14/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
/Users/arslanniazgulov/Documents/nlp_projects/ticket-routing-service/.venv/lib/python3.14/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
/Users/arslanniazgulov/Documents/nlp_projects/ticket-routing-service/.venv/lib/python3.14/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
/Users/arslanniazgulov/Documents/nlp_projects/ticket-routing-service/.venv/lib/python3.14/site-packages/torch/utils/data/dataloader.py:759: UserWarning

TrainOutput(global_step=630, training_loss=1.951277087983631, metrics={'train_runtime': 954.6164, 'train_samples_per_second': 41.912, 'train_steps_per_second': 0.66, 'total_flos': 362077777020096.0, 'train_loss': 1.951277087983631, 'epoch': 5.0})

In [74]:
preds = trainer.predict(tok_test)
preds.metrics

/Users/arslanniazgulov/Documents/nlp_projects/ticket-routing-service/.venv/lib/python3.14/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


{'test_loss': 0.3188633918762207,
 'test_macro_f1': 0.9131568393227087,
 'test_weighted_f1': 0.9131568393227086,
 'test_recall_macro': 0.9133116883116882,
 'test_precision_macro': 0.9162795935158996,
 'test_accuracy': 0.9133116883116883,
 'test_runtime': 41.229,
 'test_samples_per_second': 74.705,
 'test_steps_per_second': 1.188}

In [76]:
trainer.save_model("lora_model_de52a8ad0fff4a518f401c5917efdee7")

In [79]:
peft_model = trainer.model

# 2. Сливаем адаптер с базовой моделью прямо в памяти
merged_model = peft_model.merge_and_unload()
merged_model.config.id2label = id2label
merged_model.config.label2id = label2id

# 3. Сохраняем цельную готовую модель
merged_model.save_pretrained("final_merged_model")
tokenizer.save_pretrained("final_merged_model")

print("Готово! Полная модель сохранена в final_merged_model/")

Writing model shards: 100%|██████████| 1/1 [00:17<00:00, 17.41s/it]

Готово! Полная модель сохранена в final_merged_model/
